# Fine-tune Cross-Encoder v0.6 - Batch Size Tuning

Phase 2.2 of experimentation roadmap: Test 3 batch sizes to optimize training dynamics.

| | |
|---|---|
| **Dataset** | v0.5 — 9,350 train / 2,000 validation / 2,000 test (13,350 pairs, balanced 20% per class) |
| **Base model** | `cross-encoder/ms-marco-MiniLM-L-12-v2` |
| **Loss** | `MSELoss` |
| **Evaluator** | Spearman correlation |
| **Epochs** | **15** (fixed from Phase 1) |
| **Learning rate** | **5e-5** (optimized from Phase 2.1) |
| **Batch sizes** | 8, 16, 32 |
| **max_length** | 512 tokens |
| **Branch** | `experiment/cross-encoder-v0.6` |

**Goal**: Test if different batch sizes improve over baseline LR=5e-5, BS=16 (current 65.15%).

**Expected**: +0.5-1pp improvement → 65.5-66% target.

In [ ]:
import os
if not os.path.exists('/content/Ai-Recruiter-Mini-Ai-Service'):
    !git clone https://github.com/DangHuuLong/Ai-Recruiter-Mini-Ai-Service /content/Ai-Recruiter-Mini-Ai-Service

%cd /content/Ai-Recruiter-Mini-Ai-Service
!git checkout experiment/cross-encoder-v0.6
!git pull origin experiment/cross-encoder-v0.6

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## Helper Functions

In [ ]:
import json
import torch
import numpy as np
from scipy.stats import spearmanr
from sentence_transformers import CrossEncoder, InputExample
from typing import Any

# Evaluator: CECorrelationEvaluator (Spearman)
class CECorrelationEvaluator:
    """Evaluate cross-encoder using Spearman correlation."""
    def __init__(self, sentence_pairs: list, labels_0_1: list[float], name: str = ""):
        self.sentence_pairs = sentence_pairs
        self.labels_0_1 = labels_0_1
        self.name = name

    @classmethod
    def from_input_examples(cls, examples, name: str = "") -> "CECorrelationEvaluator":
        return cls(
            sentence_pairs=[ex.texts for ex in examples],
            labels_0_1=[ex.label for ex in examples],
            name=name,
        )

    def __call__(self, model, output_path=None, epoch: int = -1, steps: int = -1) -> float:
        preds = model.predict(self.sentence_pairs, batch_size=32, show_progress_bar=False)
        corr, _ = spearmanr(preds, self.labels_0_1)
        return float(corr) if not np.isnan(corr) else 0.0

def load_jsonl(path: str) -> list[dict[str, Any]]:
    """Load JSONL file."""
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

def load_dataset(data_dir: str) -> tuple[list[InputExample], list[InputExample], list[InputExample]]:
    """Load train/val/test InputExample lists."""
    train_data = load_jsonl(f"{data_dir}/cross_encoder_train.jsonl")
    val_data = load_jsonl(f"{data_dir}/cross_encoder_validation.jsonl")
    test_data = load_jsonl(f"{data_dir}/cross_encoder_test.jsonl")

    def to_input_examples(records):
        return [
            InputExample(texts=[rec['cv_text'], rec['jd_text']], label=rec['label'])
            for rec in records
        ]

    return to_input_examples(train_data), to_input_examples(val_data), to_input_examples(test_data)

def compute_metrics(model: CrossEncoder, examples: list[InputExample], batch_size: int = 32) -> dict:
    """Compute MAE, RMSE, LabelAcc on examples."""
    preds = model.predict([ex.texts for ex in examples], batch_size=batch_size, show_progress_bar=False)
    preds_100 = np.asarray(preds) * 100
    labels_100 = np.array([ex.label * 100 for ex in examples])

    mae = np.mean(np.abs(preds_100 - labels_100))
    rmse = np.sqrt(np.mean((preds_100 - labels_100) ** 2))
    label_acc = np.mean(np.abs(preds_100 - labels_100) <= 10)

    return {'MAE': mae, 'RMSE': rmse, 'LabelAcc': label_acc}

print("✅ Helper functions loaded.")

## Load Dataset

In [ ]:
train_examples, val_examples, test_examples = load_dataset('datasets/versions/v0.5/cross_encoder')

print(f"Train: {len(train_examples)} pairs")
print(f"Val:   {len(val_examples)} pairs")
print(f"Test:  {len(test_examples)} pairs")
print(f"Total: {len(train_examples) + len(val_examples) + len(test_examples)} pairs")

## Batch Size Tuning Grid

In [ ]:
import torch
from torch.utils.data import DataLoader
import os

# Configuration
batch_sizes = [8, 16, 32]
learning_rate = 5e-5  # Optimized from Phase 2.1
base_model = 'cross-encoder/ms-marco-MiniLM-L-12-v2'
epochs = 15

# Store all results
all_results = []

print(f"📊 Batch Size Tuning Configuration:")
print(f"  Batch sizes: {batch_sizes}")
print(f"  Learning rate: {learning_rate:.0e} (locked from Phase 2.1)")
print(f"  Epochs: {epochs}")
print(f"  Loss: MSELoss")
print(f"  Evaluator: Spearman")
print()

for bs in batch_sizes:
    print(f"\n{'='*60}")
    print(f"🚀 Training with Batch Size = {bs}")
    print(f"{'='*60}")

    run_name = f"v0.6-mse-spearman-lr5e-05-bs{bs}-15ep"
    output_dir = f"artifacts/models/cross-encoder-cv-jd-{run_name}"

    # Calculate warmup steps (10% of total)
    total_steps = (len(train_examples) // bs + 1) * epochs
    warmup_steps = int(total_steps * 0.1)

    # Initialize model
    model = CrossEncoder(
        base_model,
        num_labels=1,
        default_activation_function=torch.nn.Sigmoid()
    )

    # Setup evaluator and data
    evaluator = CECorrelationEvaluator.from_input_examples(val_examples, name="val_spearman")
    train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=bs)

    print(f"  Warmup steps: {warmup_steps} (10% of {total_steps} total)")

    # Train
    model.fit(
        train_dataloader=train_dataloader,
        evaluator=evaluator,
        epochs=epochs,
        loss_fct=torch.nn.MSELoss(),
        optimizer_params={'lr': learning_rate},
        warmup_steps=warmup_steps,
        output_path=output_dir,
        save_best_model=True,
        use_amp=True,
        max_grad_norm=1.0,
        show_progress_bar=True
    )

    # Load best model and compute metrics
    best_model = CrossEncoder(output_dir)
    val_metrics = compute_metrics(best_model, val_examples)
    test_metrics = compute_metrics(best_model, test_examples)

    print(f"\n✅ Completed BS={bs}")
    print(f"  Val LabelAcc:  {val_metrics['LabelAcc']:.4f} ({val_metrics['LabelAcc']*100:.2f}%)")
    print(f"  Test LabelAcc: {test_metrics['LabelAcc']:.4f} ({test_metrics['LabelAcc']*100:.2f}%)")

    # Store result
    result = {
        'batch_size': int(bs),
        'learning_rate': float(learning_rate),
        'run': run_name,
        'loss': 'MSE',
        'evaluator': 'Spearman',
        'epochs': epochs,
        'warmup_steps': warmup_steps,
        'total_steps': total_steps,
        'val': val_metrics,
        'test': test_metrics
    }
    all_results.append(result)

print(f"\n{'='*60}")
print(f"✅ Batch size tuning completed!")
print(f"{'='*60}")

## Results Summary

In [ ]:
import pandas as pd

# Create summary table
summary_data = []
for result in all_results:
    summary_data.append({
        'Batch Size': result['batch_size'],
        'Val LabelAcc': f"{result['val']['LabelAcc']:.4f}",
        'Test LabelAcc': f"{result['test']['LabelAcc']:.4f}",
        'Val MAE': f"{result['val']['MAE']:.4f}",
        'Test MAE': f"{result['test']['MAE']:.4f}"
    })

df = pd.DataFrame(summary_data)
print("\n📊 Batch Size Tuning Results:")
print(df.to_string(index=False))

# Find best BS
best_result = max(all_results, key=lambda x: x['test']['LabelAcc'])
best_bs = best_result['batch_size']
best_test_acc = best_result['test']['LabelAcc']

print(f"\n🏆 Best BS: {best_bs} with Test LabelAcc = {best_test_acc:.4f} ({best_test_acc*100:.2f}%)")
print(f"\n💡 Phase 2.1 Best (LR=5e-5, BS=16): 65.15%")
print(f"   Improvement: {(best_test_acc - 0.6515)*100:+.2f}pp")

## Save Reports

In [ ]:
import os
import json
from pathlib import Path

# Create reports directory
os.makedirs('artifacts/reports', exist_ok=True)

# Save individual reports
for result in all_results:
    bs = result['batch_size']
    report = {
        'base_model': 'cross-encoder/ms-marco-MiniLM-L-12-v2',
        'dataset_version': 'v0.5',
        'dataset_size': {
            'train': len(train_examples),
            'val': len(val_examples),
            'test': len(test_examples),
            'total': len(train_examples) + len(val_examples) + len(test_examples)
        },
        'run': result['run'],
        'loss': result['loss'],
        'evaluator': result['evaluator'],
        'learning_rate': result['learning_rate'],
        'batch_size': bs,
        'epochs': result['epochs'],
        'warmup_steps': result['warmup_steps'],
        'total_steps': result['total_steps'],
        'metrics': {
            'validation': {k: float(v) for k, v in result['val'].items()},
            'test': {k: float(v) for k, v in result['test'].items()}
        },
        'model_path': f'artifacts/models/cross-encoder-cv-jd-{result["run"]}'
    }

    report_path = f'artifacts/reports/fine_tune_cross_encoder_v0.6_mse_spearman_lr5e-05_bs{bs}_15ep_report.json'
    with open(report_path, 'w') as f:
        json.dump(report, f, indent=2)
    print(f"✅ Report saved: {report_path}")

# Save summary report
summary_report = {
    'experiment': 'Phase 2.2: Batch Size Tuning',
    'base_model': 'cross-encoder/ms-marco-MiniLM-L-12-v2',
    'dataset_version': 'v0.5',
    'loss': 'MSELoss',
    'evaluator': 'Spearman',
    'learning_rate': float(learning_rate),
    'epochs': epochs,
    'phase_2_1_baseline': {
        'learning_rate': 5e-5,
        'batch_size': 16,
        'test_label_acc': 0.6515
    },
    'results': all_results,
    'best': {
        'batch_size': int(best_bs),
        'learning_rate': float(learning_rate),
        'test_label_acc': float(best_test_acc),
        'improvement_over_baseline': float((best_test_acc - 0.6515)*100)
    }
}

summary_path = 'artifacts/reports/fine_tune_cross_encoder_v0.6_batch_size_tuning_report.json'
with open(summary_path, 'w') as f:
    json.dump(summary_report, f, indent=2)
print(f"\n✅ Summary report saved: {summary_path}")

## Save to Google Drive (Optional)

In [ ]:
from google.colab import drive
import shutil
import os

drive.mount('/content/drive')

drive_base = "/content/drive/MyDrive/ai-recruiter"
os.makedirs(f"{drive_base}/models", exist_ok=True)
os.makedirs(f"{drive_base}/reports", exist_ok=True)

# Save all models
for result in all_results:
    src_dir = f'artifacts/models/cross-encoder-cv-jd-{result["run"]}'
    dest_dir = f"{drive_base}/models/cross-encoder-cv-jd-{result['run']}"
    if os.path.exists(dest_dir):
        shutil.rmtree(dest_dir)
    shutil.copytree(src_dir, dest_dir)
    print(f"✅ Saved model: {result['run']}")

# Copy all reports
for file in os.listdir('artifacts/reports'):
    if 'batch_size_tuning' in file or 'bs8' in file or 'bs16' in file or 'bs32' in file:
        src = f'artifacts/reports/{file}'
        dest = f"{drive_base}/reports/{file}"
        shutil.copy(src, dest)
        print(f"✅ Saved report: {file}")

print(f"\n✅ All models and reports saved to Google Drive!")